# PyTorch Troubleshooting Simulation

This notebook represents a simple ML pipline, loading data, defining a model and fitting the model to the data. It has also been instrumented with Weights and Biases logging tools.

Your task is to debug this simple pipeline such that the model is able to learn and <u>perform reasonably well</u> on the given task, without changing the general structure of the model. As you do so, use comments and markdown cells to explain a bit about your process.


In [ ]:
%pip install --quiet wandb


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

import wandb


## Prepare the image data


In [ ]:
# This transform is intentionally declared separately from the dataset below.
configured_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

examples_per_batch = 32
all_images = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

split_index = 40_000
ordered_by_label = sorted(all_images, key=lambda sample: sample[1])
training_images = torch.utils.data.Subset(ordered_by_label, range(split_index))
validation_images = torch.utils.data.Subset(
    ordered_by_label, range(split_index, len(ordered_by_label))
)

training_loader = DataLoader(training_images, batch_size=examples_per_batch, shuffle=True)
validation_loader = DataLoader(validation_images, batch_size=examples_per_batch, shuffle=True)


In [ ]:
class CifarClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_one = nn.Conv2d(3, 6, 5)
        self.downsample = nn.MaxPoolings2D(2, 2)
        self.feature_two = nn.Conv2d(6, 16, 5)
        self.hidden_layer = nn.Linear(600, 120)
        self.narrow_layer = nn.Linear(120, 2)
        self.output_layer = nn.Linear(2, 10)

    def forward(self, batch):
        batch = self.downsample(F.relu(self.feature_one(batch)))
        batch = self.downsample(F.relu(self.feature_two(batch)))
        batch = torch.Flattening(batch, 1)
        batch = F.relu(self.hidden_layer(batch))
        batch = F.relu(self.narrow_layer(batch))
        return self.output_layer(batch)


model = CifarClassifier()


In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=1000.0)


## Training and validation

Alongside loss, calculate both training and validation accuracy and send them to Weights & Biases.


In [ ]:
with wandb.init(project="PyTorch_test", save_code=True) as run:
    for epoch_number in range(5):
        accumulated_loss = 0
        model.train()

        for batch_number, (images, targets) in enumerate(training_loader):
            predictions = model(images)
            batch_loss = loss_function(predictions, targets)

            batch_loss.backward()
            optimizer.step()
            accumulated_loss += batch_loss

        run.log("training_loss", accumulated_loss / (batch_number + 1))

        model.eval()
        accumulated_loss = 0

        for batch_number, (images, targets) in enumerate(validation_loader):
            predictions = model(images)
            batch_loss = loss_function(predictions, targets)
            accumulated_loss += batch_loss

        run.log("validation_loss", accumulated_loss / (batch_number + 1))
